# Interlock Event Log Explorer

Query the `dev-interlock-events` DynamoDB table and load results into pandas for analysis.

In [25]:
import boto3
import pandas as pd
from datetime import datetime, timezone
from boto3.dynamodb.conditions import Key, Attr

dynamodb = boto3.resource("dynamodb", region_name="ap-southeast-1")
events_table = dynamodb.Table("dev-interlock-events")
control_table = dynamodb.Table("dev-interlock-control")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

## Query Helpers

In [26]:
def query_events_by_pipeline(pipeline_id: str, limit: int = 100) -> pd.DataFrame:
    """Query all events for a pipeline, most recent first."""
    items = []
    kwargs = {
        "KeyConditionExpression": Key("PK").eq(f"PIPELINE#{pipeline_id}"),
        "ScanIndexForward": False,
        "Limit": limit,
    }
    while True:
        resp = events_table.query(**kwargs)
        items.extend(resp["Items"])
        if len(items) >= limit or "LastEvaluatedKey" not in resp:
            break
        kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]
    return _events_to_df(items[:limit])


def query_events_by_type(event_type: str, limit: int = 100) -> pd.DataFrame:
    """Query events by type using GSI1 (eventType + timestamp)."""
    items = []
    kwargs = {
        "IndexName": "GSI1",
        "KeyConditionExpression": Key("eventType").eq(event_type),
        "ScanIndexForward": False,
        "Limit": limit,
    }
    while True:
        resp = events_table.query(**kwargs)
        items.extend(resp["Items"])
        if len(items) >= limit or "LastEvaluatedKey" not in resp:
            break
        kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]
    return _events_to_df(items[:limit])


def _parse_period(period: str) -> pd.Timestamp:
    """Parse period string (20260314T04) into a UTC timestamp."""
    return pd.to_datetime(period, format="%Y%m%dT%H", utc=True)


def query_sensors(pipeline_id: str, sensor_prefix: str = "SENSOR#") -> pd.DataFrame:
    """Query sensor records from the control table."""
    resp = control_table.query(
        KeyConditionExpression=(
            Key("PK").eq(f"PIPELINE#{pipeline_id}")
            & Key("SK").begins_with(sensor_prefix)
        )
    )
    rows = []
    for item in resp["Items"]:
        row = {"PK": item["PK"], "SK": item["SK"]}
        data = item.get("data", {})
        for k, v in data.items():
            if isinstance(v, dict):
                for nested_k, nested_v in v.items():
                    row[f"{k}.{nested_k}"] = nested_v
            else:
                row[k] = v
        # Synthesize period from date+hour if not present (legacy records)
        if "period" not in row and "date" in row and "hour" in row:
            row["period"] = f"{row['date']}T{row['hour']}"
        rows.append(row)
    df = pd.DataFrame(rows)
    if "period" in df.columns:
        df["period"] = df["period"].apply(
            lambda x: _parse_period(x) if pd.notna(x) else x
        )
        df = df.sort_values("period").reset_index(drop=True)
    # Drop redundant date/hour columns
    df = df.drop(columns=[c for c in ("date", "hour") if c in df.columns])
    return df


def _events_to_df(items: list) -> pd.DataFrame:
    """Convert DynamoDB event items to a DataFrame with parsed timestamps."""
    if not items:
        return pd.DataFrame()
    df = pd.DataFrame(items)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_numeric(df["timestamp"])
        df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    cols = [c for c in ["time", "pipelineId", "eventType", "date", "scheduleId", "message"] if c in df.columns]
    cols += [c for c in df.columns if c not in cols and c not in ("PK", "SK", "ttl", "timestamp")]
    return df[cols].reset_index(drop=True)

## Timeline

Merges events (from the events table) and sensor snapshots (from the control table) into a single time-ordered view.

In [27]:
def build_timeline(pipeline_id: str, limit: int = 200) -> pd.DataFrame:
    """Build a unified timeline from events + sensor state."""
    # Events
    events_df = query_events_by_pipeline(pipeline_id, limit=limit)
    if not events_df.empty:
        event_rows = events_df.rename(columns={"time": "timestamp"})
        event_rows["source"] = "event"
        event_rows = event_rows[["timestamp", "source", "eventType", "message"]]
    else:
        event_rows = pd.DataFrame(columns=["timestamp", "source", "eventType", "message"])

    # Sensors
    sensors_df = query_sensors(pipeline_id)
    sensor_rows = []
    for _, row in sensors_df.iterrows():
        sk = row["SK"]
        sensor_name = sk.split("#")[1] if "#" in sk else sk
        updated = row.get("updatedAt")
        if pd.isna(updated):
            # postrun-baseline: use nested updatedAt
            for col in row.index:
                if col.endswith(".updatedAt") and pd.notna(row[col]):
                    updated = row[col]
                    break
        if pd.isna(updated):
            continue
        ts = pd.to_datetime(updated, utc=True)

        details = {}
        for col in row.index:
            if col in ("PK", "SK", "updatedAt", "period") or col.startswith("weather-audit."):
                continue
            if pd.notna(row[col]):
                details[col] = row[col]

        period_str = ""
        if pd.notna(row.get("period")):
            period_str = row["period"].strftime("%Y-%m-%dT%H") if hasattr(row["period"], "strftime") else str(row["period"])

        detail_parts = []
        for k in ("total_readings", "valid_readings", "batches_received", "complete"):
            if k in details:
                detail_parts.append(f"{k}={details[k]}")

        msg = f"{sensor_name} [{period_str}]: {', '.join(detail_parts)}" if detail_parts else f"{sensor_name} [{period_str}]"

        sensor_rows.append({
            "timestamp": ts,
            "source": "sensor",
            "eventType": sensor_name,
            "message": msg,
        })

    combined = pd.concat([event_rows, pd.DataFrame(sensor_rows)], ignore_index=True)
    combined = combined.sort_values("timestamp").reset_index(drop=True)
    return combined

timeline = build_timeline("dryrun-weather")
timeline

,timestamp,source,eventType,message
0,2026-03-14 04:48:19.754457+00:00,sensor,postrun-baseline,postrun-baseline []
1,2026-03-14 04:48:21.271000+00:00,event,POST_RUN_BASELINE_CAPTURED,post-run baseline captured for dryrun-weather
2,2026-03-14 04:48:21.334000+00:00,event,DRY_RUN_WOULD_TRIGGER,dry-run: would trigger dryrun-weather at 2026-03-14T04:48:21Z
3,2026-03-14 04:48:21.342000+00:00,event,DRY_RUN_SLA_PROJECTION,dry-run: SLA projection for dryrun-weather — SLA met with 52m margin
4,2026-03-14 04:48:21.351000+00:00,event,DRY_RUN_COMPLETED,dry-run: observation complete for dryrun-weather/2026-03-14T04
5,2026-03-14 04:58:19.600048+00:00,sensor,weather-audit,weather-audit [2026-03-14T04]: total_readings=14
6,2026-03-14 04:58:19.600048+00:00,sensor,weather-ready,"weather-ready [2026-03-14T04]: total_readings=14, valid_readings=11, batches_received=5, complete=True"
7,2026-03-14 04:58:21.216000+00:00,event,DRY_RUN_LATE_DATA,dry-run: late data arrived 10m after trigger point for dryrun-weather
8,2026-03-14 04:58:21.308000+00:00,event,DRY_RUN_DRIFT,dry-run: drift detected for dryrun-weather: 12 → 14 — would re-run
9,2026-03-14 05:48:19.939723+00:00,sensor,weather-audit,weather-audit [2026-03-14T05]: total_readings=11


In [28]:
df = query_events_by_pipeline("dryrun-weather")
df

,time,pipelineId,eventType,date,scheduleId,message
0,2026-03-14 04:58:21.308000+00:00,dryrun-weather,DRY_RUN_DRIFT,2026-03-14T04,stream,dry-run: drift detected for dryrun-weather: 12 → 14 — would re-run
1,2026-03-14 04:58:21.216000+00:00,dryrun-weather,DRY_RUN_LATE_DATA,2026-03-14T04,stream,dry-run: late data arrived 10m after trigger point for dryrun-weather
2,2026-03-14 04:48:21.351000+00:00,dryrun-weather,DRY_RUN_COMPLETED,2026-03-14T04,stream,dry-run: observation complete for dryrun-weather/2026-03-14T04
3,2026-03-14 04:48:21.342000+00:00,dryrun-weather,DRY_RUN_SLA_PROJECTION,2026-03-14T04,stream,dry-run: SLA projection for dryrun-weather — SLA met with 52m margin
4,2026-03-14 04:48:21.334000+00:00,dryrun-weather,DRY_RUN_WOULD_TRIGGER,2026-03-14T04,stream,dry-run: would trigger dryrun-weather at 2026-03-14T04:48:21Z
5,2026-03-14 04:48:21.271000+00:00,dryrun-weather,POST_RUN_BASELINE_CAPTURED,2026-03-14T04,stream,post-run baseline captured for dryrun-weather


In [29]:
# Event type breakdown
df["eventType"].value_counts()

eventType
DRY_RUN_DRIFT                 1
DRY_RUN_LATE_DATA             1
DRY_RUN_COMPLETED             1
DRY_RUN_SLA_PROJECTION        1
DRY_RUN_WOULD_TRIGGER         1
POST_RUN_BASELINE_CAPTURED    1
Name: count, dtype: int64

In [30]:
# Timeline by event type
df.groupby(["date", "eventType"]).size().unstack(fill_value=0)

eventType,DRY_RUN_COMPLETED,DRY_RUN_DRIFT,DRY_RUN_LATE_DATA,DRY_RUN_SLA_PROJECTION,DRY_RUN_WOULD_TRIGGER,POST_RUN_BASELINE_CAPTURED
date,,,,,,
2026-03-14T04,1,1,1,1,1,1


## Sensor State

In [31]:
sensors = query_sensors("dryrun-weather")
sensors

,PK,SK,weather-audit.period,weather-audit.total_readings,weather-audit.updatedAt,period,total_readings,updatedAt,batches_received,complete,valid_readings
0,PIPELINE#dryrun-weather,SENSOR#weather-audit#20260314T04,NaN,NaN,NaN,2026-03-14 04:00:00+00:00,14,2026-03-14T04:58:19.600048+00:00,NaN,NaN,NaN
1,PIPELINE#dryrun-weather,SENSOR#weather-ready#20260314T04,NaN,NaN,NaN,2026-03-14 04:00:00+00:00,14,2026-03-14T04:58:19.600048+00:00,5,True,11
2,PIPELINE#dryrun-weather,SENSOR#weather-audit#20260314T05,NaN,NaN,NaN,2026-03-14 05:00:00+00:00,11,2026-03-14T05:48:19.939723+00:00,NaN,NaN,NaN
3,PIPELINE#dryrun-weather,SENSOR#weather-ready#20260314T05,NaN,NaN,NaN,2026-03-14 05:00:00+00:00,11,2026-03-14T05:48:19.939723+00:00,5,NaN,10
4,PIPELINE#dryrun-weather,SENSOR#postrun-baseline#2026-03-14T04,20260314T04,12,2026-03-14T04:48:19.754457+00:00,NaT,NaN,NaN,NaN,NaN,NaN


## Query by Event Type

Common types: `DRY_RUN_WOULD_TRIGGER`, `DRY_RUN_COMPLETED`, `DRY_RUN_SLA_PROJECTION`, `DRY_RUN_DRIFT`, `DRY_RUN_LATE_DATA`, `POST_RUN_BASELINE_CAPTURED`

In [32]:
query_events_by_type("DRY_RUN_WOULD_TRIGGER")

,time,pipelineId,eventType,date,scheduleId,message
0,2026-03-14 04:48:21.334000+00:00,dryrun-weather,DRY_RUN_WOULD_TRIGGER,2026-03-14T04,stream,dry-run: would trigger dryrun-weather at 2026-03-14T04:48:21Z


## All Pipelines (scan)

In [33]:
def scan_all_events(limit: int = 500) -> pd.DataFrame:
    """Scan the full events table (use sparingly)."""
    items = []
    kwargs = {"Limit": limit}
    while True:
        resp = events_table.scan(**kwargs)
        items.extend(resp["Items"])
        if len(items) >= limit or "LastEvaluatedKey" not in resp:
            break
        kwargs["ExclusiveStartKey"] = resp["LastEvaluatedKey"]
    return _events_to_df(items[:limit])

all_events = scan_all_events()
all_events.groupby(["pipelineId", "eventType"]).size().unstack(fill_value=0)

eventType,DRY_RUN_COMPLETED,DRY_RUN_DRIFT,DRY_RUN_LATE_DATA,DRY_RUN_SLA_PROJECTION,DRY_RUN_WOULD_TRIGGER,POST_RUN_BASELINE_CAPTURED
pipelineId,,,,,,
dryrun-weather,1,1,1,1,1,1
